# 무료 LLM API 테스트: Google AI Studio & NVIDIA build & OpenRouter

본 강의의 기본 LLM 은 **로컬 Ollama(`qwen3:8b`)** 이지만, 시작 전에 **무료로 쓸 수 있는 클라우드 LLM API**
세 가지를 연결·테스트해 봅니다. 로컬 자원이 부족하거나 더 큰 모델을 잠깐 써 보고 싶을 때 유용합니다.

| 서비스 | 무료 | 키 발급 | 접속(공통 팩토리) | 예시 모델 |
|---|---|---|---|---|
| **Google AI Studio** | ✅ 무료 티어 | [aistudio.google.com](https://aistudio.google.com) | `utils.get_llm('google')` | `gemini-3.1-flash-lite` |
| **NVIDIA build** | ✅ 무료 크레딧 | [build.nvidia.com](https://build.nvidia.com) | `utils.get_llm('nvidia')` | `meta/llama-3.1-8b-instruct` 등 |
| **OpenRouter** | ✅ 무료 모델 라우터 | [openrouter.ai](https://openrouter.ai) | `utils.get_llm('openrouter')` | `openrouter/free`(무료 자동 선택) |

> 세 서비스 모두 `utils.get_llm()` 이 적절한 LangChain 커넥터(`ChatGoogleGenerativeAI` / `ChatNVIDIA` /
> `ChatOpenAI`)를 돌려주므로, 이후 노트북과 **동일한 인터페이스**로 다룰 수 있습니다.
> OpenRouter 는 OpenAI 호환이라 **OpenRouter SDK(=OpenAI SDK) 직접 호출**과 **LangChain 연결** 두 가지를 모두 보여줍니다.

> 📦 키 설정·발급 절차: [`env_guides/M02_0_free_llm_api.md`](env_guides/M02_0_free_llm_api.md)

### 전제 조건
- (선택) `notebooks/.env` 에 `GOOGLE_API_KEY`, `NVIDIA_API_KEY`, `OPENROUTER_API_KEY` 중 있는 것만 채우면 됩니다.
- 키가 없는 서비스는 오류 없이 **발급 안내만** 출력합니다.

> ⚠️ **OpenRouter 무료 모델의 사용 한도**: 신규 사용자는 테스트용 소액 무료 할당을 받습니다.
> 무료 모델은 rate limit 이 낮아(**전체 하루 50 요청**) 운영용으로는 부적합합니다.
> 크레딧을 **10 달러 이상** 충전하면 무료 모델 한도가 **하루 1,000 요청**으로 늘어납니다.
> `openrouter/free`(**Free Models Router**)를 쓰면 요청마다 무료 모델이 자동 선택됩니다.

---
## 0. 환경 설정

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(''))  # notebooks/ 를 import 경로에 추가

import utils
utils.reload_env()  # .env 재로드 (키 등 갱신)
from agentic_lib.bootstrap import to_text  # 공급자 무관 응답 정규화(list content/<think> 처리)

# 세 서비스의 커넥터 설치(이미 있으면 빠르게 통과)
#   - langchain-google-genai        : Google Gemini
#   - langchain-nvidia-ai-endpoints : NVIDIA build
#   - openai / langchain-openai     : OpenRouter(OpenAI 호환) — SDK 직접호출 + LangChain 연결
utils.uv_install(['langchain-google-genai', 'langchain-nvidia-ai-endpoints',
                  'openai', 'langchain-openai'])

# 키 설정 상태 확인(.env 에서 로드)
GOOGLE_API_KEY     = os.getenv('GOOGLE_API_KEY', '')
NVIDIA_API_KEY     = os.getenv('NVIDIA_API_KEY', '')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', '')

print('Google AI Studio 키:', '설정됨' if GOOGLE_API_KEY else '미설정 — aistudio.google.com')
print('NVIDIA build 키   :', '설정됨' if NVIDIA_API_KEY else '미설정 — build.nvidia.com')
print('OpenRouter 키     :', '설정됨' if OPENROUTER_API_KEY else '미설정 — openrouter.ai')

# 공통 테스트 프롬프트
from langchain_core.messages import HumanMessage, SystemMessage
PROMPT = [SystemMessage(content='간결하게 한국어로 답하세요.'),
          HumanMessage(content='에이전틱(Agentic) AI를 한 문장으로 설명해줘.')]

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


[uv] 설치 완료: ['langchain-google-genai', 'langchain-nvidia-ai-endpoints', 'openai', 'langchain-openai']
Google AI Studio 키: 설정됨
NVIDIA build 키   : 설정됨
OpenRouter 키     : 설정됨


---
## 1. Google AI Studio (Gemini) — 무료 티어

1. [aistudio.google.com](https://aistudio.google.com) 접속 → `Get API Key` → `Create API key`
2. `notebooks/.env` 에 `GOOGLE_API_KEY=AIza...` 저장
3. `utils.get_llm('google')` 이 `ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')` 를 반환합니다.

> Gemini 는 `content` 를 list 형식으로 주기도 하므로 `to_text()` 로 정규화해 출력합니다.

In [2]:
gemini_answer = None
if not GOOGLE_API_KEY:
    print('GOOGLE_API_KEY 미설정 — https://aistudio.google.com 에서 발급 후 .env 에 추가하세요.')
else:
    try:
        gemini = utils.get_llm('google')          # ChatGoogleGenerativeAI(gemini-3.1-flash-lite)
        resp = gemini.invoke(PROMPT)
        gemini_answer = to_text(resp.content)      # list content 정규화
        print('[Google Gemini 응답]')
        print(' ', gemini_answer)
    except Exception as e:
        print(f'[Google 호출 실패] {type(e).__name__}: {e}')

[Google Gemini 응답]
  에이전틱 AI는 단순히 명령을 수행하는 것을 넘어, 스스로 목표를 설정하고 계획을 세워 자율적으로 문제를 해결하는 지능형 시스템입니다.


---
## 2. NVIDIA build (build.nvidia.com) — 무료 크레딧

1. [build.nvidia.com](https://build.nvidia.com) 로그인 → 원하는 모델 페이지 → `Get API Key`(무료)
2. `notebooks/.env` 에 `NVIDIA_API_KEY=nvapi-...` 저장 (모델은 `NVIDIA_MODEL`, 기본 `meta/llama-3.1-8b-instruct`)
3. **`utils.get_llm('nvidia')`** 가 `ChatNVIDIA`(`langchain-nvidia-ai-endpoints`)를 반환합니다.
   다른 LangChain 모델과 동일한 `.invoke()` / `.stream()` 인터페이스를 씁니다(로컬/구글과 같은 방식).

> 사용 가능한 모델명은 build.nvidia.com 각 모델 페이지에서 확인하세요(예: `meta/llama-3.1-8b-instruct`,
> `meta/llama-3.3-70b-instruct`, `nvidia/llama-3.1-nemotron-70b-instruct`). 큰 모델은 응답이 느릴 수 있습니다.

In [3]:
# 모델은 utils/.env 의 NVIDIA_MODEL 로 설정됨(기본 meta/llama-3.1-8b-instruct)
NVIDIA_MODEL = utils.NVIDIA_MODEL

nvidia_answer = None
if not NVIDIA_API_KEY:
    print('NVIDIA_API_KEY 미설정 — https://build.nvidia.com 에서 발급 후 .env 에 추가하세요.')
else:
    try:
        nvidia = utils.get_llm('nvidia')          # ChatNVIDIA(model=NVIDIA_MODEL) 반환
        resp = nvidia.invoke(PROMPT)
        nvidia_answer = to_text(resp.content)
        print(f'[NVIDIA build 응답 · {NVIDIA_MODEL}]')
        print(' ', nvidia_answer)
    except Exception as e:
        print(f'[NVIDIA 호출 실패] {type(e).__name__}: {e}')
        print('  → 모델명이 build.nvidia.com 에서 제공되는지, 키/크레딧이 유효한지 확인하세요.')

[NVIDIA build 응답 · meta/llama-3.1-8b-instruct]
  에이전틱(Agentic) AI는 사용자의 의도와 목표를 달성하기 위해 행동하는 인공지능을 말합니다.


---
## 3. OpenRouter — 무료 모델 라우터 (`openrouter/free`)

[OpenRouter](https://openrouter.ai) 는 여러 공급자의 모델을 **하나의 OpenAI 호환 엔드포인트**
(`https://openrouter.ai/api/v1`)로 중계합니다. 신규 사용자는 소액 무료 할당을 받고, `:free` 가 붙은
**무료 모델**들을 사용할 수 있습니다.

- **`openrouter/free` (Free Models Router)** — 요청마다 사용 가능한 무료 모델을 **자동 선택**합니다.
  개별 무료 모델(예: `deepseek/deepseek-chat-v3-0324:free`)을 직접 지정할 수도 있습니다.
- **무료 한도**: 무료 모델은 rate limit 이 낮습니다 — **전체 하루 50 요청**. 크레딧을 **10달러 이상** 충전하면 **하루 1,000 요청**.
- 키는 `.env` 의 `OPENROUTER_API_KEY`(`sk-or-...`) 를 사용합니다.

OpenRouter 는 OpenAI 호환이므로 아래 **두 가지 방법**을 모두 보여줍니다.

| 방법 | 코드 | 비고 |
|---|---|---|
| **A. OpenRouter SDK** | `OpenAI(base_url='https://openrouter.ai/api/v1', api_key=...)` | OpenAI 파이썬 SDK 를 그대로 사용 |
| **B. LangChain** | `utils.get_llm('openrouter')` (= `ChatOpenAI`) | 다른 노트북과 동일한 `.invoke()`/`.stream()` 인터페이스 |

> 참고: [Free Models Router Playground](https://openrouter.ai/docs/cookbook/get-started/free-models-router-playground)

In [4]:
# 방법 A) OpenRouter SDK 직접 호출 — OpenRouter 는 OpenAI 호환이라 openai 파이썬 SDK 를 그대로 사용한다.
openrouter_answer = None
if not OPENROUTER_API_KEY:
    print('OPENROUTER_API_KEY 미설정 — https://openrouter.ai 에서 발급 후 .env 에 추가하세요.')
else:
    try:
        from openai import OpenAI
        client = OpenAI(
            base_url='https://openrouter.ai/api/v1',    # OpenRouter 엔드포인트(OpenAI 호환)
            api_key=OPENROUTER_API_KEY,
        )
        completion = client.chat.completions.create(
            model='openrouter/free',                     # 무료 모델 자동 선택 라우터(Free Models Router)
            messages=[
                {'role': 'system', 'content': '간결하게 한국어로 답하세요.'},
                {'role': 'user',   'content': '에이전틱(Agentic) AI를 한 문장으로 설명해줘.'},
            ],
            # (선택) openrouter.ai 랭킹/사용 통계에 앱을 표기하고 싶을 때만 지정
            extra_headers={'HTTP-Referer': 'https://localhost', 'X-Title': 'Agentic AI Tutorial'},
        )
        openrouter_answer = completion.choices[0].message.content
        # 라우터가 실제로 고른 무료 모델명은 응답의 model 필드로 확인 가능
        print(f'[OpenRouter SDK 응답 · 실제 선택 모델: {completion.model}]')
        print(' ', openrouter_answer)
    except Exception as e:
        print(f'[OpenRouter SDK 호출 실패] {type(e).__name__}: {e}')
        print('  → 키/무료 한도(무료 모델 하루 50요청, 10크레딧 이상 시 1000요청)를 확인하세요.')

[OpenRouter SDK 호출 실패] RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1785024000000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day

In [5]:
# 방법 B) LangChain 연결 — utils.get_llm('openrouter') 가 ChatOpenAI(base_url=OpenRouter) 를 반환.
#          이후 노트북과 동일한 .invoke() / .stream() 인터페이스로 다룰 수 있다.
openrouter_lc_answer = None
if not OPENROUTER_API_KEY:
    print('OPENROUTER_API_KEY 미설정 — LangChain 예시를 건너뜁니다.')
else:
    try:
        openrouter_llm = utils.get_llm('openrouter')   # ChatOpenAI(model='openrouter/free', base_url=...)
        resp = openrouter_llm.invoke(PROMPT)
        openrouter_lc_answer = to_text(resp.content)   # 응답 정규화
        print(f'[OpenRouter LangChain 응답 · {utils.OPENROUTER_MODEL}]')
        print(' ', openrouter_lc_answer)
    except Exception as e:
        print(f'[OpenRouter LangChain 호출 실패] {type(e).__name__}: {e}')

[OpenRouter LangChain 호출 실패] RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1785024000000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests p

---
## 4. 스트리밍 — Google · NVIDIA · OpenRouter **공통 함수**

`utils.get_llm(provider)` 가 돌려주는 LangChain 모델은 공급자와 무관하게 `.stream()` 을 지원합니다
(`ChatGoogleGenerativeAI`·`ChatNVIDIA`·`ChatOpenAI` 모두 `.stream()`/`.astream()` 보유). 따라서 **하나의
함수**로 Google·NVIDIA·OpenRouter 모두 토큰 스트리밍을 볼 수 있습니다.

- **reasoning(사고) 모델**(예: NVIDIA `nvidia/nemotron-...`)은 `additional_kwargs['reasoning_content']` 로
  사고 과정을 함께 스트리밍합니다(아래 함수가 둘 다 처리). Gemini 에는 해당 없음.
- 청크 `content` 는 공급자별로 str/list 차이가 있어 `to_text()` 로 정규화합니다. 단 **스트리밍에서는
  `strip=False`** 를 줘야 합니다 — NVIDIA/OpenRouter 는 **토큰 단위** 스트리밍이라 토큰 앞 공백(`" 사용자"`)까지
  기본 strip 에 잘려 단어가 붙어버리기 때문입니다(Gemini 는 큰 청크라 영향이 적음).

In [6]:
def stream_answer(provider, messages=PROMPT):
    """주어진 공급자로 응답을 토큰 스트리밍으로 출력한다(Google/NVIDIA/OpenRouter 공통)."""
    llm = utils.get_llm(provider)                 # 공급자 무관 LangChain 모델
    for chunk in llm.stream(messages):
        ak = getattr(chunk, "additional_kwargs", None) or {}
        if ak.get("reasoning_content"):            # reasoning 모델의 사고 과정
            print(ak["reasoning_content"], end="")
        # 스트리밍은 청크(토큰)마다 출력하므로 strip=False 로 공백을 보존한다.
        #   NVIDIA/OpenRouter 는 토큰 단위 스트리밍이라, 기본 strip 이면 토큰 앞 공백(" 사용자")까지
        #   잘려 '단어가붙어버리는' 현상이 생긴다. Gemini 는 큰 청크 단위라 영향이 적다.
        print(to_text(chunk.content, strip_thinking=False, strip=False), end="")  # 답변 토큰
    print()

# 키가 설정된 서비스만 스트리밍
targets = []
if GOOGLE_API_KEY:
    targets.append(("google", "Google Gemini"))
if NVIDIA_API_KEY:
    targets.append(("nvidia", f"NVIDIA {utils.NVIDIA_MODEL}"))
if OPENROUTER_API_KEY:
    targets.append(("openrouter", f"OpenRouter {utils.OPENROUTER_MODEL}"))

for provider, label in targets:
    print(f"\n[스트리밍 · {label}]")
    try:
        stream_answer(provider)
    except Exception as e:
        print(f"  (스트리밍 실패) {type(e).__name__}: {e}")
if not targets:
    print("설정된 키가 없어 스트리밍 예시를 건너뜁니다.")


[스트리밍 · Google Gemini]


에이전틱 AI는 단순히 명령을 수행하는 것을 넘어, 스스로 목표를 설정하고 계획을 세워 자율적으로 문제를 해결하는 지능형 시스템입니다.

[스트리밍 · NVIDIA meta/llama-3.1-8b-instruct]


에이전틱(Agentic) AI는 사용자의 의도와 목표를 달성하기 위해 행동하는 인공지능을 말합니다.



[스트리밍 · OpenRouter openrouter/free]


  (스트리밍 실패) RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1785024000000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code'

---
## 5. 같은 질문 · 서비스별 최종 응답 비교

동일 프롬프트에 대한 세 무료 서비스의 (앞 §1·§2·§3에서 받은) 응답을 나란히 봅니다(설정된 서비스만).
OpenRouter 는 LangChain 경로(§3-B)의 응답을 사용합니다.

In [7]:
print('질문:', PROMPT[-1].content)
print('=' * 70)
comparison = [
    ('Google Gemini', gemini_answer),
    (f'NVIDIA {NVIDIA_MODEL}', nvidia_answer),
    (f'OpenRouter {utils.OPENROUTER_MODEL}', openrouter_lc_answer),
]
for name, ans in comparison:
    print(f'\n[{name}]')
    print(' ', ans if ans else '(키 미설정/호출 실패로 건너뜀)')

질문: 에이전틱(Agentic) AI를 한 문장으로 설명해줘.

[Google Gemini]
  에이전틱 AI는 단순히 명령을 수행하는 것을 넘어, 스스로 목표를 설정하고 계획을 세워 자율적으로 문제를 해결하는 지능형 시스템입니다.

[NVIDIA meta/llama-3.1-8b-instruct]
  에이전틱(Agentic) AI는 사용자의 의도와 목표를 달성하기 위해 행동하는 인공지능을 말합니다.

[OpenRouter openrouter/free]
  (키 미설정/호출 실패로 건너뜀)


---
## 정리

| 서비스 | `.env` 키 | 접속 코드 |
|---|---|---|
| **Google AI Studio** | `GOOGLE_API_KEY` | `utils.get_llm('google')` (= `ChatGoogleGenerativeAI`) |
| **NVIDIA build** | `NVIDIA_API_KEY` (+`NVIDIA_MODEL`) | `utils.get_llm('nvidia')` (= `ChatNVIDIA`) |
| **OpenRouter** | `OPENROUTER_API_KEY` (+`OPENROUTER_MODEL`) | SDK: `OpenAI(base_url=...)` · LangChain: `utils.get_llm('openrouter')` (= `ChatOpenAI`) |

- 세 서비스 모두 **무료**로 시작할 수 있고, `utils.get_llm()` 으로 다른 모델과 동일하게(`.invoke()` / `.stream()`) 다룹니다.
- 응답 형식 차이(Gemini 의 list content 등)는 `agentic_lib.bootstrap.to_text()` 로 흡수합니다.
- **OpenRouter** 는 OpenAI 호환이라 **OpenRouter SDK(OpenAI SDK) 직접 호출**과 **LangChain(`ChatOpenAI`)** 둘 다 가능합니다.
  기본 모델 `openrouter/free`(Free Models Router)가 무료 모델을 자동 선택하며, 무료 한도는 하루 50요청(10크레딧 이상 시 1,000요청)입니다.
- 본 강의 기본은 **로컬 Ollama**(`M02_1_local_llm`)이며, `.env` 의 `LLM_PROVIDER` 한 줄로
  로컬 ↔ 클라우드(`google`/`nvidia`/`openrouter`)를 전환합니다. NVIDIA build 는 NAT(`notebooks/NAT_Tutorial/` 폴더)에서 `_type: nim` 으로도 사용합니다.

### 참고
- Google AI Studio: https://aistudio.google.com
- NVIDIA build: https://build.nvidia.com  ·  LangChain 커넥터: `langchain-nvidia-ai-endpoints`
- OpenRouter: https://openrouter.ai  ·  Free Models Router: https://openrouter.ai/docs/cookbook/get-started/free-models-router-playground